# FraudLens — Production ML Pipeline for Transaction Fraud Detection

**Dataset:** IEEE-CIS Fraud Detection (Kaggle) — 590K e-commerce transactions, 394 features, 3.5% fraud rate

**Stack:** pandas · XGBoost · LightGBM · SHAP · MLflow · FastAPI · Evidently · Docker

---

## Notebook Structure

| Section | What it covers |
|---------|----------------|
| 0 | Setup & Kaggle data download |
| 1 | EDA — class imbalance, missingness, distributions |
| 2 | Feature Engineering — velocity, temporal, frequency encoding |
| 3 | Preprocessing pipeline — imputation, encoding, scaling |
| 4 | Model Training — Logistic Regression, XGBoost, LightGBM |
| 5 | Experiment Tracking — MLflow logging |
| 6 | Evaluation — AUC-ROC, PR-AUC, F1, confusion matrix |
| 7 | Threshold Tuning — precision-recall tradeoff |
| 8 | SHAP Explainability — summary, force, waterfall plots |
| 9 | Model Export — save best model for serving |
| 10 | Drift Simulation — Evidently drift report |
| 11 | Resume Bullets Generator |

## Section 0 — Setup & Data Download

In [ ]:
# Install dependencies
!pip install kaggle lightgbm xgboost shap mlflow evidently scikit-learn pandas numpy matplotlib seaborn joblib --quiet

In [ ]:
# ─── Kaggle Setup ───────────────────────────────────────────────────────────
# 1. Go to kaggle.com → Account → Create New API Token → downloads kaggle.json
# 2. Place kaggle.json at ~/.kaggle/kaggle.json
# 3. Accept competition rules at: https://www.kaggle.com/c/ieee-fraud-detection

import os
import subprocess

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Download competition data
subprocess.run([
    "kaggle", "competitions", "download",
    "-c", "ieee-fraud-detection",
    "-p", DATA_DIR
], check=True)

subprocess.run(["unzip", "-q", f"{DATA_DIR}/ieee-fraud-detection.zip", "-d", DATA_DIR], check=True)
print("Files:", os.listdir(DATA_DIR))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import json
import time
from pathlib import Path

# ML
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    roc_curve, precision_recall_curve, classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
import shap
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

# Plotting config
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
FRAUD_COLOR = "#e74c3c"
LEGIT_COLOR = "#2ecc71"
ACCENT     = "#2c3e50"

print("✓ All imports loaded")

In [ ]:
# Load data
train_txn  = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_id   = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
test_txn   = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_id    = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

# Merge on TransactionID
train = train_txn.merge(train_id, on="TransactionID", how="left")
test  = test_txn.merge(test_id,  on="TransactionID", how="left")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Fraud rate  : {train['isFraud'].mean():.4f} ({train['isFraud'].sum():,} fraud / {len(train):,} total)")

## Section 1 — Exploratory Data Analysis

In [ ]:
# ─── 1.1 Class imbalance ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = train["isFraud"].value_counts()
axes[0].bar(["Legit", "Fraud"], counts.values,
            color=[LEGIT_COLOR, FRAUD_COLOR], edgecolor="white", linewidth=1.5)
axes[0].set_title("Class Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Transaction Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2000, f"{v:,}\n({v/len(train)*100:.1f}%)",
                 ha="center", fontsize=10, fontweight="bold")

# Transaction amount by class
train.groupby("isFraud")["TransactionAmt"].apply(
    lambda x: np.log1p(x)
).unstack().T.plot.hist(ax=axes[1], bins=60, alpha=0.7,
                        color=[LEGIT_COLOR, FRAUD_COLOR], edgecolor="none")
axes[1].set_title("log(TransactionAmt) by Class", fontsize=13, fontweight="bold")
axes[1].set_xlabel("log(1 + Amount)")
axes[1].legend(["Legit", "Fraud"])

plt.suptitle("IEEE-CIS: Class Imbalance & Amount Distribution",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plots/01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Class imbalance ratio: {counts[0]/counts[1]:.1f}:1")

In [ ]:
# ─── 1.2 Missingness heatmap ─────────────────────────────────────────────────
os.makedirs("plots", exist_ok=True)

miss_pct = (train.isnull().mean() * 100).sort_values(ascending=False)
high_miss = miss_pct[miss_pct > 50]

print(f"Features with >50% missing: {len(high_miss)}")
print(f"Features with >80% missing: {(miss_pct > 80).sum()}")

fig, ax = plt.subplots(figsize=(14, 5))
top_miss = miss_pct.head(50)
colors = [FRAUD_COLOR if v > 50 else "#f39c12" if v > 20 else LEGIT_COLOR
          for v in top_miss.values]
ax.barh(range(len(top_miss)), top_miss.values, color=colors, edgecolor="none")
ax.set_yticks(range(len(top_miss)))
ax.set_yticklabels(top_miss.index, fontsize=7)
ax.axvline(50, color="red", linestyle="--", alpha=0.7, label="50% threshold")
ax.set_xlabel("Missing %")
ax.set_title("Top 50 Features by Missingness", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("plots/02_missingness.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── 1.3 Temporal pattern: fraud rate over time ──────────────────────────────
# TransactionDT is seconds from a reference point
train["day"] = train["TransactionDT"] // 86400
fraud_by_day = train.groupby("day")["isFraud"].agg(["mean", "sum", "count"]).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()
ax1.fill_between(fraud_by_day["day"], fraud_by_day["mean"] * 100,
                 color=FRAUD_COLOR, alpha=0.4, label="Fraud Rate %")
ax1.plot(fraud_by_day["day"], fraud_by_day["mean"] * 100, color=FRAUD_COLOR, lw=1.5)
ax2.plot(fraud_by_day["day"], fraud_by_day["count"],
         color=ACCENT, lw=1.2, linestyle="--", alpha=0.7, label="Total Txns")
ax1.set_xlabel("Day (relative)")
ax1.set_ylabel("Fraud Rate (%)", color=FRAUD_COLOR)
ax2.set_ylabel("Transaction Volume", color=ACCENT)
ax1.set_title("Daily Fraud Rate & Transaction Volume Over Time",
              fontsize=13, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.savefig("plots/03_temporal_fraud.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── 1.4 Product code & card type fraud rates ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col in zip(axes, ["ProductCD", "card4"]):
    rates = train.groupby(col)["isFraud"].agg(["mean", "count"]).reset_index()
    rates = rates.sort_values("mean", ascending=False)
    bars = ax.bar(rates[col].astype(str), rates["mean"] * 100,
                  color=[FRAUD_COLOR if v > 0.05 else LEGIT_COLOR for v in rates["mean"]],
                  edgecolor="white")
    ax.set_title(f"Fraud Rate by {col}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Fraud Rate (%)")
    ax.set_xlabel(col)
    for bar, (_, row) in zip(bars, rates.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                f"{row['count']:,}", ha="center", fontsize=8, color="gray")

plt.suptitle("Fraud Rate by Categorical Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/04_categorical_fraud_rates.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 2 — Feature Engineering

This is the most important section. Most kaggle notebooks skip straight to `model.fit()`. We build meaningful features.

In [ ]:
def engineer_features(df):
    """
    Full feature engineering pipeline for IEEE-CIS fraud detection.
    Adds temporal, velocity, and aggregation features.
    """
    df = df.copy()

    # ── Temporal features ────────────────────────────────────────────────────
    # TransactionDT is seconds elapsed from a reference point
    df["hour"]    = (df["TransactionDT"] // 3600) % 24   # hour of day
    df["day"]     = (df["TransactionDT"] // 86400) % 7   # day of week
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 5)).astype(int)
    df["is_weekend"] = (df["day"] >= 5).astype(int)

    # ── D-column engineering ─────────────────────────────────────────────────
    # D-columns represent days since some event (card use, signup, etc.)
    # Normalise: subtract D1 (days since card use started) to get relative time
    for col in [f"D{i}" for i in range(2, 16) if f"D{i}" in df.columns]:
        if "D1" in df.columns:
            df[f"{col}_norm"] = df[col] - df["D1"]

    # ── Transaction amount features ──────────────────────────────────────────
    df["TransactionAmt_log"]     = np.log1p(df["TransactionAmt"])
    df["TransactionAmt_decimal"] = df["TransactionAmt"] - df["TransactionAmt"].astype(int)
    # Round-number transactions are often suspicious
    df["is_round_amount"] = (df["TransactionAmt_decimal"] == 0).astype(int)

    # ── Card aggregation features ────────────────────────────────────────────
    # Group by card identifiers — velocity signals
    card_cols = ["card1", "card2", "card3", "card5"]
    for col in card_cols:
        if col in df.columns:
            # Transaction count per card (frequency)
            freq = df[col].map(df[col].value_counts())
            df[f"{col}_freq"] = freq
            # Mean transaction amount per card
            mean_amt = df.groupby(col)["TransactionAmt"].transform("mean")
            df[f"{col}_mean_amt"] = mean_amt
            # Std of transaction amount per card
            std_amt = df.groupby(col)["TransactionAmt"].transform("std")
            df[f"{col}_std_amt"] = std_amt
            # Ratio: this transaction vs card mean (outlier signal)
            df[f"{col}_amt_ratio"] = df["TransactionAmt"] / (mean_amt + 1e-5)

    # ── Email domain features ────────────────────────────────────────────────
    # P_emaildomain = purchaser, R_emaildomain = recipient
    for col in ["P_emaildomain", "R_emaildomain"]:
        if col in df.columns:
            # Extract domain suffix (e.g. .com, .net)
            df[f"{col}_suffix"] = df[col].str.split(".").str[-1]
            # Frequency encode email domain
            freq = df[col].map(df[col].value_counts())
            df[f"{col}_freq"] = freq

    # Match between purchaser and recipient email domains
    if "P_emaildomain" in df.columns and "R_emaildomain" in df.columns:
        df["email_domain_match"] = (
            df["P_emaildomain"] == df["R_emaildomain"]
        ).astype(int)

    # ── Device & browser frequency ───────────────────────────────────────────
    for col in ["DeviceType", "DeviceInfo"]:
        if col in df.columns:
            df[f"{col}_freq"] = df[col].map(df[col].value_counts())

    # ── Address match ────────────────────────────────────────────────────────
    # addr1/addr2 are billing address components
    if "addr1" in df.columns and "addr2" in df.columns:
        df["addr_match"] = (df["addr1"] == df["addr2"]).astype(int)

    # ── C-column sum (counts of addresses, cards seen with card) ─────────────
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    if c_cols:
        df["C_sum"] = df[c_cols].sum(axis=1)
        df["C_mean"] = df[c_cols].mean(axis=1)

    # ── V-column summary stats ───────────────────────────────────────────────
    # V columns are Vesta's proprietary features — encode as summary stats
    v_cols = [c for c in df.columns if c.startswith("V") and c[1:].isdigit()]
    if v_cols:
        df["V_sum"]  = df[v_cols].sum(axis=1)
        df["V_mean"] = df[v_cols].mean(axis=1)
        df["V_std"]  = df[v_cols].std(axis=1)
        df["V_null_count"] = df[v_cols].isnull().sum(axis=1)

    print(f"✓ Feature engineering complete: {df.shape[1]} features (was {train.shape[1]})")
    return df


train_fe = engineer_features(train)
test_fe  = engineer_features(test)

## Section 3 — Preprocessing Pipeline

In [ ]:
# ─── 3.1 Drop high-missingness features ─────────────────────────────────────
TARGET = "isFraud"
DROP_COLS = ["TransactionID", "TransactionDT", TARGET]

# Drop columns with >80% missing — not recoverable
miss_pct_fe = train_fe.isnull().mean()
high_miss_cols = miss_pct_fe[miss_pct_fe > 0.8].index.tolist()
print(f"Dropping {len(high_miss_cols)} columns with >80% missing")

drop_all = DROP_COLS + high_miss_cols
feature_cols = [c for c in train_fe.columns if c not in drop_all]

X = train_fe[feature_cols].copy()
y = train_fe[TARGET].copy()
X_test = test_fe[[c for c in feature_cols if c in test_fe.columns]].copy()

print(f"Feature count after filtering: {X.shape[1]}")

In [ ]:
# ─── 3.2 Label encode categoricals, impute, scale ───────────────────────────
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features: {len(cat_cols)}")
print(f"Numeric features    : {len(num_cols)}")

# Label encode all categoricals
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Combine train+test for consistent encoding
    combined = pd.concat([X[col], X_test[col] if col in X_test.columns
                          else pd.Series()], ignore_index=True).astype(str)
    le.fit(combined)
    X[col]  = le.transform(X[col].astype(str))
    if col in X_test.columns:
        X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# Median imputation for numeric (preserves skew better than mean for fraud data)
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X[num_cols]  = num_imputer.fit_transform(X[num_cols])
X[cat_cols]  = cat_imputer.fit_transform(X[cat_cols])

if not X_test.empty:
    test_num_cols = [c for c in num_cols if c in X_test.columns]
    test_cat_cols = [c for c in cat_cols if c in X_test.columns]
    X_test[test_num_cols] = num_imputer.transform(X_test[test_num_cols])
    X_test[test_cat_cols] = cat_imputer.transform(X_test[test_cat_cols])

print("✓ Imputation complete")
print(f"Remaining nulls in X: {X.isnull().sum().sum()}")

In [ ]:
# ─── 3.3 Train / Validation split (time-aware) ──────────────────────────────
# CRITICAL: Use temporal split, NOT random split
# Random split leaks future data into training — AUC will be inflated
# Use the last 20% of time as validation

train_idx = train_fe.index[train_fe["TransactionDT"] <= train_fe["TransactionDT"].quantile(0.8)]
val_idx   = train_fe.index[train_fe["TransactionDT"] >  train_fe["TransactionDT"].quantile(0.8)]

X_train, X_val = X.loc[train_idx], X.loc[val_idx]
y_train, y_val = y.loc[train_idx], y.loc[val_idx]

# Class weight for imbalance
fraud_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

print(f"Train size      : {len(X_train):,} ({y_train.mean():.3%} fraud)")
print(f"Validation size : {len(X_val):,} ({y_val.mean():.3%} fraud)")
print(f"Class weight    : {fraud_ratio:.1f}:1")

## Section 4 — Model Training

In [ ]:
# ─── 4.1 Evaluation helper ──────────────────────────────────────────────────
def evaluate_model(name, model, X_val, y_val, threshold=0.5):
    """Returns a metrics dict for a fitted model."""
    proba = model.predict_proba(X_val)[:, 1]
    preds = (proba >= threshold).astype(int)

    metrics = {
        "model"     : name,
        "roc_auc"   : roc_auc_score(y_val, proba),
        "pr_auc"    : average_precision_score(y_val, proba),
        "f1"        : f1_score(y_val, preds),
        "precision" : precision_score(y_val, preds, zero_division=0),
        "recall"    : recall_score(y_val, preds),
        "threshold" : threshold,
    }
    return metrics, proba

all_results = []
all_probas  = {}

In [ ]:
# ─── 4.2 Baseline: Logistic Regression ──────────────────────────────────────
print("Training Logistic Regression (baseline)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

lr = LogisticRegression(
    class_weight="balanced",
    max_iter=500,
    solver="saga",
    C=0.1,
    random_state=42
)
t0 = time.time()
lr.fit(X_train_scaled, y_train)
lr_time = time.time() - t0

# Wrap to use unscaled interface
class ScaledModel:
    def __init__(self, model, scaler):
        self.model, self.scaler = model, scaler
    def predict_proba(self, X):
        return self.model.predict_proba(self.scaler.transform(X))

lr_wrapped = ScaledModel(lr, scaler)
lr_metrics, lr_proba = evaluate_model("Logistic Regression", lr_wrapped, X_val, y_val)
lr_metrics["train_time_s"] = lr_time
all_results.append(lr_metrics)
all_probas["Logistic Regression"] = lr_proba

print(f"  ROC-AUC : {lr_metrics['roc_auc']:.4f}")
print(f"  PR-AUC  : {lr_metrics['pr_auc']:.4f}")
print(f"  F1      : {lr_metrics['f1']:.4f}")
print(f"  Time    : {lr_time:.1f}s")

In [ ]:
# ─── 4.3 XGBoost ────────────────────────────────────────────────────────────
print("Training XGBoost...")
xgb_params = {
    "n_estimators"    : 500,
    "max_depth"       : 6,
    "learning_rate"   : 0.05,
    "subsample"       : 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": fraud_ratio,   # handle imbalance
    "eval_metric"     : "auc",
    "early_stopping_rounds": 50,
    "random_state"    : 42,
    "n_jobs"          : -1,
    "tree_method"     : "hist",        # faster
}

xgb_model = xgb.XGBClassifier(**xgb_params)
t0 = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)
xgb_time = time.time() - t0

xgb_metrics, xgb_proba = evaluate_model("XGBoost", xgb_model, X_val, y_val)
xgb_metrics["train_time_s"] = xgb_time
xgb_metrics["best_iteration"] = xgb_model.best_iteration
all_results.append(xgb_metrics)
all_probas["XGBoost"] = xgb_proba

print(f"\n  ROC-AUC        : {xgb_metrics['roc_auc']:.4f}")
print(f"  PR-AUC         : {xgb_metrics['pr_auc']:.4f}")
print(f"  F1             : {xgb_metrics['f1']:.4f}")
print(f"  Best iteration : {xgb_model.best_iteration}")
print(f"  Time           : {xgb_time:.1f}s")

In [ ]:
# ─── 4.4 LightGBM ────────────────────────────────────────────────────────────
print("Training LightGBM...")
lgb_params = {
    "n_estimators"    : 1000,
    "max_depth"       : -1,
    "num_leaves"      : 63,
    "learning_rate"   : 0.05,
    "subsample"       : 0.8,
    "colsample_bytree": 0.8,
    "min_child_samples": 20,
    "scale_pos_weight": fraud_ratio,
    "is_unbalance"    : False,
    "random_state"    : 42,
    "n_jobs"          : -1,
    "verbose"         : -1,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
t0 = time.time()
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100)
    ]
)
lgb_time = time.time() - t0

lgb_metrics, lgb_proba = evaluate_model("LightGBM", lgb_model, X_val, y_val)
lgb_metrics["train_time_s"] = lgb_time
lgb_metrics["best_iteration"] = lgb_model.best_iteration_
all_results.append(lgb_metrics)
all_probas["LightGBM"] = lgb_proba

print(f"\n  ROC-AUC        : {lgb_metrics['roc_auc']:.4f}")
print(f"  PR-AUC         : {lgb_metrics['pr_auc']:.4f}")
print(f"  F1             : {lgb_metrics['f1']:.4f}")
print(f"  Best iteration : {lgb_model.best_iteration_}")
print(f"  Time           : {lgb_time:.1f}s")

## Section 5 — MLflow Experiment Tracking

In [ ]:
# ─── 5.1 Log all runs to MLflow ─────────────────────────────────────────────
# Start MLflow UI with: mlflow ui --port 5000
# Then open: http://localhost:5000

mlflow.set_experiment("FraudLens")

models_to_log = [
    ("Logistic Regression", lr, lr_params := {"C": 0.1, "solver": "saga"},         lr_metrics),
    ("XGBoost",             xgb_model, xgb_params,                                  xgb_metrics),
    ("LightGBM",            lgb_model, lgb_params,                                  lgb_metrics),
]

run_ids = {}
for name, model, params, metrics in models_to_log:
    with mlflow.start_run(run_name=name) as run:
        # Log params
        mlflow.log_params({k: v for k, v in params.items()
                           if isinstance(v, (int, float, str, bool))})

        # Log metrics
        mlflow.log_metrics({
            "roc_auc"   : metrics["roc_auc"],
            "pr_auc"    : metrics["pr_auc"],
            "f1"        : metrics["f1"],
            "precision" : metrics["precision"],
            "recall"    : metrics["recall"],
            "train_time": metrics.get("train_time_s", 0),
        })

        # Log dataset info
        mlflow.log_params({
            "train_size"   : len(X_train),
            "val_size"     : len(X_val),
            "feature_count": X_train.shape[1],
            "fraud_rate"   : float(y_train.mean()),
        })

        # Log model artifact
        if name == "XGBoost":
            mlflow.xgboost.log_model(model, "model")
        elif name == "LightGBM":
            mlflow.lightgbm.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(lr, "model")

        run_ids[name] = run.info.run_id
        print(f"  ✓ Logged: {name} | ROC-AUC={metrics['roc_auc']:.4f} | Run: {run.info.run_id[:8]}")

print(f"\n→ View experiments: mlflow ui --port 5000")

## Section 6 — Evaluation Dashboard

In [ ]:
# ─── 6.1 Model comparison table ─────────────────────────────────────────────
results_df = pd.DataFrame(all_results).set_index("model")
print("\n" + "="*65)
print("MODEL COMPARISON")
print("="*65)
print(results_df[["roc_auc", "pr_auc", "f1", "precision", "recall", "train_time_s"]]
      .round(4).to_string())
print("="*65)

best_model_name = results_df["roc_auc"].idxmax()
print(f"\n→ Best model: {best_model_name} (ROC-AUC={results_df.loc[best_model_name,'roc_auc']:.4f})")

In [ ]:
# ─── 6.2 ROC + PR curves for all models ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#3498db", "#e74c3c", "#2ecc71"]

for (name, proba), color in zip(all_probas.items(), colors):
    # ROC curve
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc_val = roc_auc_score(y_val, proba)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc_val:.4f})")

    # PR curve
    prec, rec, _ = precision_recall_curve(y_val, proba)
    pr_auc_val = average_precision_score(y_val, proba)
    axes[1].plot(rec, prec, color=color, lw=2, label=f"{name} (AP={pr_auc_val:.4f})")

# ROC plot formatting
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (0.5)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves", fontsize=13, fontweight="bold")
axes[0].legend(loc="lower right", fontsize=9)

# PR plot formatting
baseline = y_val.mean()
axes[1].axhline(baseline, color="gray", linestyle="--", alpha=0.5,
                label=f"Random ({baseline:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves", fontsize=13, fontweight="bold")
axes[1].legend(loc="upper right", fontsize=9)

plt.suptitle("Model Performance Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/05_roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── 6.3 Confusion matrix for best model ────────────────────────────────────
best_proba = all_probas[best_model_name]
best_preds = (best_proba >= 0.5).astype(int)

cm = confusion_matrix(y_val, best_preds)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
            xticklabels=["Legit", "Fraud"],
            yticklabels=["Legit", "Fraud"],
            linewidths=0.5, linecolor="white")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_model_name}",
             fontsize=13, fontweight="bold")

# Add percentage annotations
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.7, f"({cm_pct[i,j]:.1f}%)",
                ha="center", va="center", fontsize=9, color="gray")

plt.tight_layout()
plt.savefig("plots/06_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(y_val, best_preds, target_names=["Legit", "Fraud"]))

## Section 7 — Threshold Tuning

In [ ]:
# ─── 7.1 Find optimal threshold ─────────────────────────────────────────────
# Default 0.5 is rarely optimal for imbalanced fraud data
# We optimize for F1 but show the precision-recall tradeoff

thresholds = np.linspace(0.01, 0.99, 200)
f1_scores  = []
prec_scores = []
rec_scores  = []

for t in thresholds:
    preds = (best_proba >= t).astype(int)
    f1_scores.append(f1_score(y_val, preds, zero_division=0))
    prec_scores.append(precision_score(y_val, preds, zero_division=0))
    rec_scores.append(recall_score(y_val, preds, zero_division=0))

best_thresh_idx = np.argmax(f1_scores)
OPTIMAL_THRESHOLD = thresholds[best_thresh_idx]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(thresholds, f1_scores,   color="#9b59b6", lw=2.5, label="F1")
ax.plot(thresholds, prec_scores, color="#3498db", lw=1.5, label="Precision", alpha=0.8)
ax.plot(thresholds, rec_scores,  color="#e74c3c", lw=1.5, label="Recall", alpha=0.8)
ax.axvline(OPTIMAL_THRESHOLD, color="black", linestyle="--",
           label=f"Optimal threshold = {OPTIMAL_THRESHOLD:.3f}")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Tuning — {best_model_name}", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("plots/07_threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

# Re-evaluate with optimal threshold
tuned_metrics, _ = evaluate_model(
    f"{best_model_name} (tuned)",
    {"LightGBM": lgb_model, "XGBoost": xgb_model}.get(best_model_name, lr_wrapped),
    X_val, y_val, threshold=OPTIMAL_THRESHOLD
)
print(f"\nOptimal threshold : {OPTIMAL_THRESHOLD:.3f}")
print(f"F1 at 0.5        : {f1_scores[100]:.4f}")
print(f"F1 at optimal    : {tuned_metrics['f1']:.4f}")
print(f"Precision        : {tuned_metrics['precision']:.4f}")
print(f"Recall           : {tuned_metrics['recall']:.4f}")

## Section 8 — SHAP Explainability

In [ ]:
# ─── 8.1 Compute SHAP values ────────────────────────────────────────────────
# Use 2000-sample subset for speed; SHAP is O(n * features)
print("Computing SHAP values (this takes ~2-5 minutes)...")

SHAP_SAMPLE = 2000
X_shap = X_val.sample(SHAP_SAMPLE, random_state=42)

best_model_obj = {"LightGBM": lgb_model, "XGBoost": xgb_model}.get(
    best_model_name, lgb_model
)

explainer   = shap.TreeExplainer(best_model_obj)
shap_values = explainer.shap_values(X_shap)

# For LightGBM binary classification, shap_values is a list [class0, class1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]   # fraud class
else:
    shap_vals = shap_values

print(f"✓ SHAP values computed | shape: {shap_vals.shape}")

In [ ]:
# ─── 8.2 SHAP Summary Plot (Beeswarm) ───────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals, X_shap,
    max_display=20,
    show=False,
    plot_type="dot"
)
plt.title(f"SHAP Feature Importance — {best_model_name}",
          fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("plots/08_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── 8.3 SHAP Bar Plot (mean absolute SHAP values) ──────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_vals, X_shap,
    max_display=20,
    show=False,
    plot_type="bar"
)
plt.title(f"Top 20 Features by Mean |SHAP| — {best_model_name}",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/09_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── 8.4 SHAP Force Plot — single transaction explanation ───────────────────
# Pick a high-confidence fraud prediction to explain
fraud_proba_shap = best_model_obj.predict_proba(X_shap)[:, 1]
fraud_idx = np.argmax(fraud_proba_shap)   # highest fraud probability

print(f"Explaining transaction at index {fraud_idx}")
print(f"Fraud probability: {fraud_proba_shap[fraud_idx]:.4f}")
print(f"Actual label     : {y_val.iloc[fraud_idx]}")

force_plot = shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list)
    else explainer.expected_value,
    shap_vals[fraud_idx],
    X_shap.iloc[fraud_idx],
    matplotlib=True,
    show=False,
    figsize=(18, 3)
)
plt.title(f"SHAP Force Plot — High-Risk Transaction (prob={fraud_proba_shap[fraud_idx]:.3f})",
          fontsize=11, pad=30)
plt.tight_layout()
plt.savefig("plots/10_shap_force.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── 8.5 SHAP Waterfall Plot ─────────────────────────────────────────────────
shap_exp = shap.Explanation(
    values=shap_vals[fraud_idx],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
               else explainer.expected_value,
    data=X_shap.iloc[fraud_idx].values,
    feature_names=X_shap.columns.tolist()
)

plt.figure(figsize=(10, 8))
shap.waterfall_plot(shap_exp, max_display=15, show=False)
plt.title(f"SHAP Waterfall — Top 15 Drivers for Fraud Prediction",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/11_shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── 8.6 Extract top SHAP features for API ──────────────────────────────────
mean_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.Series(mean_shap, index=X_shap.columns)\
                    .sort_values(ascending=False)

TOP_FEATURES = shap_importance.head(50).index.tolist()
print("Top 10 fraud signals by SHAP:")
for i, (feat, val) in enumerate(shap_importance.head(10).items(), 1):
    print(f"  {i:2d}. {feat:<35} mean|SHAP|={val:.4f}")

# Save for API
os.makedirs("models", exist_ok=True)
json.dump(TOP_FEATURES, open("models/top_features.json", "w"))

## Section 9 — Model Export

In [ ]:
# ─── 9.1 Save best model and metadata ───────────────────────────────────────
best_model_obj = {"LightGBM": lgb_model, "XGBoost": xgb_model}.get(
    best_model_name, lr_wrapped
)

joblib.dump(best_model_obj,  "models/best_model.joblib")
joblib.dump(num_imputer,     "models/num_imputer.joblib")
joblib.dump(cat_imputer,     "models/cat_imputer.joblib")
joblib.dump(label_encoders,  "models/label_encoders.joblib")
joblib.dump(feature_cols,    "models/feature_cols.joblib")

# Save model metadata
model_meta = {
    "model_name"        : best_model_name,
    "roc_auc"           : round(float(results_df.loc[best_model_name, "roc_auc"]), 4),
    "pr_auc"            : round(float(results_df.loc[best_model_name, "pr_auc"]), 4),
    "f1"                : round(float(tuned_metrics["f1"]), 4),
    "optimal_threshold" : round(float(OPTIMAL_THRESHOLD), 4),
    "train_size"        : int(len(X_train)),
    "val_size"          : int(len(X_val)),
    "feature_count"     : int(X_train.shape[1]),
    "fraud_rate_train"  : round(float(y_train.mean()), 4),
    "top_features"      : TOP_FEATURES[:10],
    "mlflow_run_id"     : run_ids.get(best_model_name, ""),
}
json.dump(model_meta, open("models/model_meta.json", "w"), indent=2)

print("Saved artifacts:")
for f in Path("models").iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
print(f"\nBest model  : {best_model_name}")
print(f"ROC-AUC     : {model_meta['roc_auc']}")
print(f"Threshold   : {model_meta['optimal_threshold']}")

## Section 10 — Drift Detection with Evidently

In [ ]:
!pip install evidently --quiet

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset
from evidently.metrics import ColumnDriftMetric

# ─── 10.1 Simulate production drift ──────────────────────────────────────────
# Take val set as "production" data and inject drift into 3 key features
ref_data  = X_train.sample(5000, random_state=42).copy()
prod_data = X_val.sample(2000, random_state=99).copy()

# Inject drift: shift TransactionAmt_log up by 0.5 std (simulate higher-value txns)
drift_features = ["TransactionAmt_log", "card1_freq", "V_mean"]
for feat in drift_features:
    if feat in prod_data.columns:
        prod_data[feat] = prod_data[feat] + prod_data[feat].std() * 0.8

print(f"Reference data  : {ref_data.shape}")
print(f"Production data : {prod_data.shape}")
print(f"Drifted features: {drift_features}")

In [ ]:
# ─── 10.2 Generate Evidently drift report ───────────────────────────────────
# Use top 20 features for the report (full 400 is slow)
report_features = TOP_FEATURES[:20]

drift_report = Report(metrics=[
    DataDriftPreset(),
])

drift_report.run(
    reference_data=ref_data[report_features],
    current_data=prod_data[report_features]
)

os.makedirs("reports", exist_ok=True)
drift_report.save_html("reports/drift_report.html")
print("✓ Drift report saved → reports/drift_report.html")
print("  Open this file in a browser to explore feature drift")

# Print drift summary
drift_result = drift_report.as_dict()
n_drifted = drift_result["metrics"][0]["result"]["number_of_drifted_columns"]
n_total   = drift_result["metrics"][0]["result"]["number_of_columns"]
print(f"\nDrift summary: {n_drifted}/{n_total} features drifted")
print(f"Share drifted: {n_drifted/n_total*100:.1f}%")